# Trabalho 2 — Exploração inicial do *dataset* DESCMERC / ELEVEN

Base de descrições de mercadorias extraídas de Notas Fiscais Eletrônicas (NFe), rotuladas por auditores fiscais segundo a Nomenclatura Comum do Mercosul (NCM).

- Repositório: https://github.com/vinidiol/descmerc
- Artigo: Di Oliveira, Weigang e Rocha Filho (2022), *ELEVEN Data-Set: A Labeled Set of Descriptions of Goods Captured from Brazilian Electronic Invoices*, WEBIST 2022.

Colunas da base completa (`BaseDESC_NCM`):
- `XPROD`: descrição da mercadoria na NFe (texto livre, até 120 caracteres)
- `NCM`: código NCM declarado na NFe
- `rotulo`: especificação da mercadoria identificada pelos especialistas, segundo a NCM
- `Item`: código do rótulo atribuído pelos especialistas (classe)

In [1]:
%pip install itables

Note: you may need to restart the kernel to use updated packages.


In [2]:
#@title Importa o dataset completo
import os
import pandas as pd
from itables import show

# Lê do clone local, se existir; senão, do GitHub do autor do dataset
LOCAL = 'descmerc/'
REMOTO = 'https://github.com/vinidiol/descmerc/raw/main/'
ROOT = LOCAL if os.path.exists(LOCAL + 'BaseDESC_NCM.zip') else REMOTO

# Arquivo separado por tabulação; o zip contém um único CSV.
# NCM é lido como texto, para não ser tratado como número.
base = pd.read_csv(
    ROOT + 'BaseDESC_NCM.zip',
    sep='\t',
    dtype={'NCM': str},
    )

# Tabela de códigos de Item, conforme o README do repositório
ITENS = {
    30: 'Cachaça e destilados',
    31: 'Vinhos, vermutes, sidras e outras fermentadas',
    38: 'Perfumaria, higiene pessoal e cosméticos',
    39: 'Materiais de limpeza',
    40: 'Produtos alimentícios',
    41: 'Materiais de construção',
    42: 'Material elétrico',
}
base['Item'] = base['Item'].astype('category')

print(f'Origem: {ROOT}')
print(f'Dimensões: {base.shape}')

Origem: descmerc/
Dimensões: (1117623, 4)


## Visão geral

In [3]:
base.info()

<class 'pandas.DataFrame'>
RangeIndex: 1117623 entries, 0 to 1117622
Data columns (total 4 columns):
 #   Column  Non-Null Count    Dtype   
---  ------  --------------    -----   
 0   XPROD   1117623 non-null  str     
 1   NCM     1117623 non-null  str     
 2   rotulo  1117623 non-null  str     
 3   Item    1117623 non-null  category
dtypes: category(1), str(3)
memory usage: 26.6 MB


In [4]:
show(base)

In [5]:
#@title Valores ausentes e valores distintos por coluna
pd.DataFrame({
    'ausentes': base.isna().sum(),
    'distintos': base.nunique(),
})

,ausentes,distintos
XPROD,0,1071512
NCM,0,2174
rotulo,0,362
Item,0,7


In [6]:
#@title Registros por Item
contagem = base['Item'].value_counts().rename('registros').to_frame()
contagem['descrição'] = contagem.index.map(ITENS)
contagem['fração'] = contagem['registros'] / len(base)
contagem

,registros,descrição,fração
Item,,,
41,325301,Materiais de construção,0.291065
40,296539,Produtos alimentícios,0.265330
38,260359,"Perfumaria, higiene pessoal e cosméticos",0.232958
42,89637,Material elétrico,0.080203
39,84110,Materiais de limpeza,0.075258
31,58422,"Vinhos, vermutes, sidras e outras fermentadas",0.052273
30,3255,Cachaça e destilados,0.002912


In [7]:
#@title Comprimento das descrições (XPROD), em caracteres
base['XPROD'].str.len().describe()

count    1.117623e+06
mean     3.760824e+01
std      9.739804e+00
min      1.000000e+00
25%      3.100000e+01
50%      3.700000e+01
75%      4.300000e+01
max      1.200000e+02
Name: XPROD, dtype: float64

In [8]:
#@title Comprimento dos códigos NCM, em caracteres
base['NCM'].str.len().value_counts()

NCM
8    1077550
7      39915
1        134
2         24
Name: count, dtype: int64

## Amostra usada no artigo

`AmostraDescMerc.csv`: 15.000 descrições, 5.000 de cada uma das classes 38 (cosméticos), 39 (limpeza) e 40 (alimentos), com colunas `Text` e `Category`. É o arquivo usado nos *notebooks* do repositório (`ELEVEN_Data_Set_R_Models.ipynb` e, na versão *one-hot*, `ELEVEN_Data_Set_bert_model.ipynb`).

In [9]:
#@title Importa a amostra
amostra = pd.read_csv(ROOT + 'AmostraDescMerc.csv', sep='\t')
print(f'Dimensões: {amostra.shape}')
amostra['Category'].value_counts()

Dimensões: (15000, 2)


Category
cosmeticos    5000
matlimpeza    5000
alimentos     5000
Name: count, dtype: int64

In [10]:
show(amostra)